# DroneAId — Inference Pipeline

**Pipeline lengkap** untuk sistem triase otomatis korban bencana berbasis UAV.  
Notebook ini memuat model, menjalankan deteksi, analisis pergerakan, estimasi pose, klasifikasi status korban, segmentasi terrain, dan pathfinding — seluruhnya berjalan **offline di CPU**.

### Pipeline Flow
```
Frame(s) masuk
  │
  ├─[1] YOLOv12s Detection (SARD) → bbox manusia
  ├─[2] Farneback Optical Flow → deteksi gerakan per bbox
  ├─[3] Morphological Pose → aspect ratio bbox (prone/standing)
  ├─[4] Status Classification (rule-based) → triage level
  ├─[5] YOLO11s-seg Segmentation (RescueNet) → terrain map
  ├─[6] World Map Buffer → GPS + triage per korban
  └─[7] A* Pathfinding → rute evakuasi optimal
```

### Constraint Compliance
| Constraint | Limit | Actual |
|---|---|---|
| C-A1 (Size) | ≤ 50 MB | 37.494 MB |
| C-A2 (CPU-only) | ✓ | ONNX Runtime CPUExecutionProvider |
| C-A3 (Latency) | ≤ 3s | Det 1.79s + Seg 0.32s |
| C-A4 (Framework) | ONNX Runtime | ✓ |
| C-A5 (Offline) | Fully offline | ✓ |

## 0. Setup & Dependencies

In [ ]:
import os
import sys
import time
import json
import heapq
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import cv2
import onnxruntime as ort

# Lock CPU threads (match i5 Gen 8: 4 physical cores)
THREADS = 4
os.environ['OMP_NUM_THREADS'] = str(THREADS)
os.environ['MKL_NUM_THREADS'] = str(THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(THREADS)

print(f'onnxruntime: {ort.__version__}')
print(f'opencv: {cv2.__version__}')
print(f'numpy: {np.__version__}')

## 1. Konfigurasi Model & Path

In [ ]:
# === MODEL PATHS ===
# Sesuaikan path berikut jika menjalankan di environment berbeda
DET_MODEL_PATH = '../models/yolov12s_sard_best.onnx'      # 18.093 MB
SEG_MODEL_PATH = '../models/yolo11s_seg_rescuenet_best.onnx'  # 19.401 MB

# === INFERENCE CONFIG ===
DET_IMGSZ = 1280       # Detection input resolution
SEG_IMGSZ = 640        # Segmentation input resolution
DET_CONF_THRESH = 0.25 # Detection confidence threshold
DET_IOU_THRESH = 0.45  # NMS IoU threshold
FLOW_MAG_THRESH = 2.0  # Optical flow magnitude threshold (pixels)

# === TERRAIN CLASSES (RescueNet) ===
SEG_CLASSES = [
    'water', 'building-no-damage', 'building-minor-damage',
    'building-major-damage', 'building-total-destruction',
    'road-clear', 'road-blocked', 'vehicle'
]
SAFE_TERRAIN = {'road-clear', 'building-no-damage'}
DANGER_TERRAIN = {'water', 'road-blocked', 'building-total-destruction', 'building-major-damage'}

# Validasi file model
for p in [DET_MODEL_PATH, SEG_MODEL_PATH]:
    sz = os.path.getsize(p) / 1024**2
    print(f'{Path(p).name}: {sz:.3f} MB')
total = sum(os.path.getsize(p)/1024**2 for p in [DET_MODEL_PATH, SEG_MODEL_PATH])
print(f'Total model size: {total:.3f} MB (limit: 50 MB) — {"PASS" if total <= 50 else "FAIL"}')

## 2. Load ONNX Models

In [ ]:
def make_session(path: str, threads: int = THREADS) -> ort.InferenceSession:
    """Load ONNX model dengan CPU-only provider."""
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = threads
    opts.inter_op_num_threads = 1
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, opts, providers=['CPUExecutionProvider'])

t0 = time.perf_counter()
det_sess = make_session(DET_MODEL_PATH)
seg_sess = make_session(SEG_MODEL_PATH)
print(f'Models loaded in {time.perf_counter()-t0:.3f}s')

det_input_name = det_sess.get_inputs()[0].name
seg_input_name = seg_sess.get_inputs()[0].name
print(f'Detection input: {det_sess.get_inputs()[0].shape}')
print(f'Segmentation input: {seg_sess.get_inputs()[0].shape}')

## 3. Pre-processing Helpers

In [ ]:
def preprocess(img_bgr: np.ndarray, imgsz: int) -> tuple[np.ndarray, float, tuple[int,int]]:
    """Letterbox resize + normalize untuk YOLO ONNX input.
    
    Returns:
        blob: (1, 3, imgsz, imgsz) float32 [0,1]
        ratio: scale factor
        pad: (pad_w, pad_h) in pixels
    """
    h, w = img_bgr.shape[:2]
    ratio = min(imgsz / h, imgsz / w)
    new_w, new_h = int(w * ratio), int(h * ratio)
    pad_w, pad_h = (imgsz - new_w) // 2, (imgsz - new_h) // 2
    
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((imgsz, imgsz, 3), 114, dtype=np.uint8)
    canvas[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
    
    blob = canvas.astype(np.float32) / 255.0
    blob = blob.transpose(2, 0, 1)[np.newaxis]  # HWC -> 1CHW
    return blob, ratio, (pad_w, pad_h)

## 4. Stage 1 — Deteksi Manusia (YOLOv12s + SARD)

In [ ]:
def nms(boxes, scores, iou_thresh):
    """Non-Maximum Suppression sederhana."""
    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)
        inds = np.where(iou <= iou_thresh)[0]
        order = order[inds + 1]
    return np.array(keep)


def detect_humans(img_bgr: np.ndarray) -> list[dict]:
    """Deteksi manusia menggunakan YOLOv12s ONNX.
    
    Returns:
        List of {'bbox': [x1,y1,x2,y2], 'conf': float}
    """
    blob, ratio, (pw, ph) = preprocess(img_bgr, DET_IMGSZ)
    outputs = det_sess.run(None, {det_input_name: blob})
    preds = outputs[0]  # shape: (1, N, 5) — [cx, cy, w, h, conf]
    
    if preds.ndim == 3:
        preds = preds[0]
    # Handle transposed output: (1, 5, N) -> (N, 5)
    if preds.shape[0] == 5 and preds.shape[1] > 5:
        preds = preds.T
    
    # Filter by confidence
    if preds.shape[1] == 5:
        scores = preds[:, 4]
    elif preds.shape[1] == 6:
        scores = preds[:, 4] * preds[:, 5]
    else:
        # Multi-class: columns 4+ are class scores
        scores = preds[:, 4:].max(axis=1)
    
    mask = scores > DET_CONF_THRESH
    preds = preds[mask]
    scores = scores[mask]
    
    if len(preds) == 0:
        return []
    
    # Convert cx,cy,w,h -> x1,y1,x2,y2 (letterbox coords)
    cx, cy, w, h = preds[:,0], preds[:,1], preds[:,2], preds[:,3]
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    boxes = np.stack([x1, y1, x2, y2], axis=1)
    
    # NMS
    keep = nms(boxes, scores, DET_IOU_THRESH)
    boxes = boxes[keep]
    scores = scores[keep]
    
    # Map back to original image coords
    results = []
    for box, score in zip(boxes, scores):
        bx1 = (box[0] - pw) / ratio
        by1 = (box[1] - ph) / ratio
        bx2 = (box[2] - pw) / ratio
        by2 = (box[3] - ph) / ratio
        oh, ow = img_bgr.shape[:2]
        bx1, by1 = max(0, bx1), max(0, by1)
        bx2, by2 = min(ow, bx2), min(oh, by2)
        results.append({
            'bbox': [float(bx1), float(by1), float(bx2), float(by2)],
            'conf': float(score),
        })
    return results

print(f'detect_humans() ready — conf={DET_CONF_THRESH}, iou={DET_IOU_THRESH}')

## 5. Stage 2 — Optical Flow (Farneback)

In [ ]:
class MotionAnalyzer:
    """Analisis pergerakan antar frame menggunakan Farneback Optical Flow.
    
    Untuk setiap bounding box korban, hitung rata-rata magnitude flow.
    Jika > threshold → 'moving', else → 'static'.
    """
    def __init__(self, mag_thresh: float = FLOW_MAG_THRESH):
        self.prev_gray = None
        self.mag_thresh = mag_thresh
    
    def update(self, frame_bgr: np.ndarray) -> Optional[np.ndarray]:
        """Update dengan frame baru. Return flow magnitude map atau None jika frame pertama."""
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        if self.prev_gray is None:
            self.prev_gray = gray
            return None
        
        flow = cv2.calcOpticalFlowFarneback(
            self.prev_gray, gray,
            flow=None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2,
            flags=0
        )
        self.prev_gray = gray
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        return mag
    
    def classify_motion(self, mag_map: np.ndarray, bbox: list[float]) -> tuple[str, float]:
        """Klasifikasi gerakan dalam bounding box.
        
        Returns:
            ('moving'|'static', avg_magnitude)
        """
        x1, y1, x2, y2 = [int(v) for v in bbox]
        h, w = mag_map.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        
        if x2 <= x1 or y2 <= y1:
            return 'static', 0.0
        
        roi = mag_map[y1:y2, x1:x2]
        avg_mag = float(np.mean(roi))
        label = 'moving' if avg_mag > self.mag_thresh else 'static'
        return label, avg_mag

print('MotionAnalyzer ready — Farneback Optical Flow')

## 6. Stage 3 — Morphological Pose Estimation

In [ ]:
def estimate_pose(bbox: list[float]) -> tuple[str, float]:
    """Estimasi pose berdasarkan aspect ratio bounding box.
    
    Logika:
        w/h > 1.0 → korban terbaring (prone)
        w/h ≤ 1.0 → korban berdiri/duduk (standing)
    
    Returns:
        ('prone'|'standing', aspect_ratio)
    """
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    if h < 1e-6:
        return 'prone', float('inf')
    ar = w / h
    pose = 'prone' if ar > 1.0 else 'standing'
    return pose, float(ar)

print('estimate_pose() ready — aspect ratio method')

## 7. Stage 4 — Status Classification (Rule-Based Triage)

In [ ]:
# Triage priority: lower = more urgent
TRIAGE_TABLE = {
    ('static',  'prone'):    ('critical',   1),  # Tidak bergerak + terbaring → paling darurat
    ('static',  'standing'): ('needs_help', 2),  # Tidak bergerak + berdiri → butuh bantuan
    ('moving',  'prone'):    ('injured',    3),  # Bergerak + terbaring → cedera tapi sadar
    ('moving',  'standing'): ('normal',     4),  # Bergerak + berdiri → kondisi baik
}

TRIAGE_COLORS = {
    'critical':   (0, 0, 255),    # Merah
    'needs_help': (0, 165, 255),  # Oranye
    'injured':    (0, 255, 255),  # Kuning
    'normal':     (0, 255, 0),    # Hijau
}


def classify_status(motion: str, pose: str) -> tuple[str, int]:
    """Klasifikasi status korban berdasarkan gerakan + pose.
    
    Returns:
        (status_label, priority) — priority 1 = paling darurat
    """
    return TRIAGE_TABLE.get((motion, pose), ('unknown', 5))

print('Triage classification ready')
for (m, p), (s, pri) in TRIAGE_TABLE.items():
    print(f'  {m:8s} + {p:8s} → {s:10s} (priority {pri})')

## 8. Stage 5 — Segmentasi Terrain (YOLO11s-seg + RescueNet)

In [ ]:
def segment_terrain(img_bgr: np.ndarray) -> tuple[np.ndarray, dict]:
    """Segmentasi terrain menggunakan YOLO11s-seg ONNX.
    
    Returns:
        class_map: (H, W) array of class indices
        stats: dict with area percentages per class
    """
    blob, ratio, (pw, ph) = preprocess(img_bgr, SEG_IMGSZ)
    outputs = seg_sess.run(None, {seg_input_name: blob})
    
    # YOLO seg output: [0] = detection, [1] = proto masks
    det_out = outputs[0]  # (1, N, 4+1+num_cls+mask_dim)
    if len(outputs) > 1:
        proto = outputs[1]  # (1, mask_dim, mh, mw)
    else:
        # Fallback: no segmentation masks available
        h, w = img_bgr.shape[:2]
        return np.zeros((h, w), dtype=np.int32), {}
    
    if det_out.ndim == 3:
        det_out = det_out[0]
    if det_out.shape[0] < det_out.shape[1]:
        det_out = det_out.T
    
    proto = proto[0]  # (mask_dim, mh, mw)
    mask_dim = proto.shape[0]
    num_cls = len(SEG_CLASSES)
    
    # Parse: [cx, cy, w, h, cls_scores..., mask_coeffs...]
    cls_scores = det_out[:, 4:4+num_cls]
    mask_coeffs = det_out[:, 4+num_cls:4+num_cls+mask_dim]
    
    best_cls = cls_scores.max(axis=1)
    valid = best_cls > DET_CONF_THRESH
    
    h, w = img_bgr.shape[:2]
    class_map = np.full((SEG_IMGSZ, SEG_IMGSZ), -1, dtype=np.int32)
    
    if valid.sum() > 0:
        coeffs = mask_coeffs[valid]          # (K, mask_dim)
        classes = cls_scores[valid].argmax(axis=1)  # (K,)
        scores = best_cls[valid]
        
        # Compute instance masks: coeffs @ proto -> (K, mh, mw)
        masks = coeffs @ proto.reshape(mask_dim, -1)  # (K, mh*mw)
        mh, mw = proto.shape[1], proto.shape[2]
        masks = masks.reshape(-1, mh, mw)
        masks = 1 / (1 + np.exp(-masks))  # sigmoid
        
        # Resize masks to input size and assign class
        order = scores.argsort()  # low confidence first, overwritten by high
        for idx in order:
            m = cv2.resize(masks[idx], (SEG_IMGSZ, SEG_IMGSZ), interpolation=cv2.INTER_LINEAR)
            class_map[m > 0.5] = int(classes[idx])
    
    # Crop letterbox padding and resize to original
    new_h, new_w = int(h * ratio), int(w * ratio)
    cropped = class_map[ph:ph+new_h, pw:pw+new_w]
    if cropped.size == 0:
        class_map_orig = np.full((h, w), -1, dtype=np.int32)
    else:
        class_map_orig = cv2.resize(cropped, (w, h), interpolation=cv2.INTER_NEAREST)
    
    # Compute area stats
    total_px = class_map_orig.size
    stats = {}
    for i, cls_name in enumerate(SEG_CLASSES):
        count = int((class_map_orig == i).sum())
        if count > 0:
            stats[cls_name] = round(count / total_px * 100, 2)
    
    return class_map_orig, stats

print(f'segment_terrain() ready — {len(SEG_CLASSES)} classes')

## 9. Stage 6 — World Map Buffer

In [ ]:
@dataclass
class Victim:
    """Data korban yang terdeteksi."""
    victim_id: int
    bbox: list[float]
    conf: float
    motion: str         # 'moving' | 'static'
    motion_mag: float
    pose: str           # 'prone' | 'standing'
    aspect_ratio: float
    status: str         # 'critical' | 'needs_help' | 'injured' | 'normal'
    priority: int       # 1 = most urgent
    gps: Optional[tuple[float, float]] = None  # (lat, lon) jika tersedia


@dataclass
class WorldMapBuffer:
    """Buffer peta dunia — menyimpan semua korban & terrain dari seluruh frame."""
    victims: list[Victim] = field(default_factory=list)
    terrain_stats: dict = field(default_factory=dict)
    frame_count: int = 0
    
    def add_victim(self, v: Victim):
        self.victims.append(v)
    
    def update_terrain(self, stats: dict):
        self.terrain_stats = stats
        self.frame_count += 1
    
    def get_sorted_victims(self) -> list[Victim]:
        """Return victims sorted by priority (most urgent first)."""
        return sorted(self.victims, key=lambda v: (v.priority, -v.conf))
    
    def summary(self) -> dict:
        counts = {}
        for v in self.victims:
            counts[v.status] = counts.get(v.status, 0) + 1
        return {
            'total_victims': len(self.victims),
            'by_status': counts,
            'terrain': self.terrain_stats,
            'frames_processed': self.frame_count,
        }

print('WorldMapBuffer ready')

## 10. Stage 7 — A* Pathfinding

In [ ]:
def build_traversability_grid(class_map: np.ndarray, grid_size: int = 50) -> np.ndarray:
    """Konversi terrain class map ke grid traversability.
    
    Returns:
        grid: (grid_size, grid_size) — 0=traversable, 1=blocked
    """
    h, w = class_map.shape
    grid = np.zeros((grid_size, grid_size), dtype=np.uint8)
    cell_h, cell_w = h // grid_size, w // grid_size
    
    danger_ids = {i for i, name in enumerate(SEG_CLASSES) if name in DANGER_TERRAIN}
    
    for gy in range(grid_size):
        for gx in range(grid_size):
            cell = class_map[gy*cell_h:(gy+1)*cell_h, gx*cell_w:(gx+1)*cell_w]
            danger_ratio = sum((cell == d).sum() for d in danger_ids) / max(cell.size, 1)
            if danger_ratio > 0.5:
                grid[gy, gx] = 1  # blocked
    
    return grid


def astar(grid: np.ndarray, start: tuple[int,int], goal: tuple[int,int]) -> Optional[list[tuple[int,int]]]:
    """A* pathfinding pada grid traversability.
    
    Args:
        grid: 0=free, 1=blocked
        start: (row, col)
        goal: (row, col)
    
    Returns:
        List of (row, col) waypoints, or None if no path found.
    """
    rows, cols = grid.shape
    
    def heuristic(a, b):
        return abs(a[0]-b[0]) + abs(a[1]-b[1])  # Manhattan distance
    
    if grid[start[0], start[1]] == 1 or grid[goal[0], goal[1]] == 1:
        return None
    
    open_set = [(heuristic(start, goal), 0, start)]
    came_from = {}
    g_score = {start: 0}
    closed = set()
    
    neighbors = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    
    while open_set:
        _, cost, current = heapq.heappop(open_set)
        
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]
        
        if current in closed:
            continue
        closed.add(current)
        
        for dr, dc in neighbors:
            nr, nc = current[0]+dr, current[1]+dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 0:
                move_cost = 1.414 if (dr != 0 and dc != 0) else 1.0
                new_g = g_score[current] + move_cost
                if new_g < g_score.get((nr, nc), float('inf')):
                    g_score[(nr, nc)] = new_g
                    f = new_g + heuristic((nr, nc), goal)
                    came_from[(nr, nc)] = current
                    heapq.heappush(open_set, (f, new_g, (nr, nc)))
    
    return None  # No path found

print('A* pathfinding ready — 8-directional movement')

## 11. Integrated Pipeline

In [ ]:
def run_pipeline(
    frames: list[np.ndarray],
    sar_start: tuple[int,int] = (0, 0),
    grid_size: int = 50,
) -> dict:
    """Jalankan pipeline end-to-end pada sequence of frames.
    
    Args:
        frames: list of BGR images
        sar_start: posisi awal tim SAR dalam grid coords
        grid_size: resolusi grid untuk pathfinding
    
    Returns:
        dict dengan hasil pipeline lengkap
    """
    world = WorldMapBuffer()
    motion = MotionAnalyzer()
    victim_id = 0
    timings = {'detection': [], 'segmentation': [], 'flow': [], 'total': []}
    
    for fi, frame in enumerate(frames):
        t_total = time.perf_counter()
        print(f'\n--- Frame {fi+1}/{len(frames)} ---')
        
        # [1] Detection
        t = time.perf_counter()
        detections = detect_humans(frame)
        dt_det = time.perf_counter() - t
        timings['detection'].append(dt_det)
        print(f'  Detection: {len(detections)} humans ({dt_det:.3f}s)')
        
        # [2] Optical Flow
        t = time.perf_counter()
        mag_map = motion.update(frame)
        dt_flow = time.perf_counter() - t
        timings['flow'].append(dt_flow)
        
        # [3-4] Pose + Status per detection
        for det in detections:
            bbox = det['bbox']
            
            # Motion
            if mag_map is not None:
                mot, mag = motion.classify_motion(mag_map, bbox)
            else:
                mot, mag = 'static', 0.0  # First frame: assume static
            
            # Pose
            pose, ar = estimate_pose(bbox)
            
            # Status
            status, priority = classify_status(mot, pose)
            
            v = Victim(
                victim_id=victim_id,
                bbox=bbox, conf=det['conf'],
                motion=mot, motion_mag=mag,
                pose=pose, aspect_ratio=ar,
                status=status, priority=priority,
            )
            world.add_victim(v)
            victim_id += 1
            print(f'    Victim #{v.victim_id}: {status} (conf={det["conf"]:.2f}, motion={mot}, pose={pose}, ar={ar:.2f})')
        
        # [5] Segmentation
        t = time.perf_counter()
        class_map, terrain_stats = segment_terrain(frame)
        dt_seg = time.perf_counter() - t
        timings['segmentation'].append(dt_seg)
        world.update_terrain(terrain_stats)
        print(f'  Segmentation: {terrain_stats} ({dt_seg:.3f}s)')
        
        dt_total = time.perf_counter() - t_total
        timings['total'].append(dt_total)
        print(f'  Frame total: {dt_total:.3f}s')
    
    # [6] World Map Summary
    print('\n=== WORLD MAP SUMMARY ===')
    summary = world.summary()
    print(json.dumps(summary, indent=2))
    
    # [7] A* Pathfinding — route to most critical victim
    sorted_victims = world.get_sorted_victims()
    pathfinding_results = []
    
    if sorted_victims and class_map is not None:
        grid = build_traversability_grid(class_map, grid_size)
        blocked_pct = grid.sum() / grid.size * 100
        print(f'\nTraversability grid: {grid_size}x{grid_size}, {blocked_pct:.1f}% blocked')
        
        h, w = frames[-1].shape[:2]
        for v in sorted_victims[:3]:  # Top 3 most critical
            cx = (v.bbox[0] + v.bbox[2]) / 2
            cy = (v.bbox[1] + v.bbox[3]) / 2
            goal = (int(cy / h * grid_size), int(cx / w * grid_size))
            goal = (min(goal[0], grid_size-1), min(goal[1], grid_size-1))
            
            path = astar(grid, sar_start, goal)
            if path:
                print(f'  Route to Victim #{v.victim_id} ({v.status}): {len(path)} waypoints')
                pathfinding_results.append({
                    'victim_id': v.victim_id,
                    'status': v.status,
                    'priority': v.priority,
                    'path_length': len(path),
                    'path': path[:5],  # First 5 waypoints only
                })
            else:
                print(f'  No path to Victim #{v.victim_id} ({v.status}) — area blocked')
    
    return {
        'world_map': summary,
        'victims': [asdict(v) for v in sorted_victims],
        'pathfinding': pathfinding_results,
        'timings': {k: [round(x,4) for x in v] for k,v in timings.items()},
    }

print('run_pipeline() ready')

## 12. Visualization

In [ ]:
def visualize_result(frame: np.ndarray, victims: list[Victim],
                     class_map: Optional[np.ndarray] = None,
                     path: Optional[list] = None,
                     grid_size: int = 50) -> np.ndarray:
    """Visualisasi hasil pipeline pada frame."""
    vis = frame.copy()
    
    # Draw terrain overlay
    if class_map is not None:
        terrain_colors = {
            0: (255, 100, 100),  # water - biru
            1: (100, 200, 100),  # building-no-damage - hijau
            2: (100, 200, 200),  # building-minor - kuning
            3: (50, 100, 200),   # building-major - oranye
            4: (50, 50, 200),    # building-total-destruction - merah
            5: (200, 200, 200),  # road-clear - abu
            6: (100, 100, 150),  # road-blocked - coklat
            7: (200, 150, 50),   # vehicle - cyan
        }
        overlay = np.zeros_like(vis)
        for cls_id, color in terrain_colors.items():
            overlay[class_map == cls_id] = color
        mask = class_map >= 0
        vis[mask] = cv2.addWeighted(vis, 0.6, overlay, 0.4, 0)[mask]
    
    # Draw victim boxes with triage colors
    for v in victims:
        x1, y1, x2, y2 = [int(c) for c in v.bbox]
        color = TRIAGE_COLORS.get(v.status, (255,255,255))
        cv2.rectangle(vis, (x1,y1), (x2,y2), color, 2)
        
        label = f'#{v.victim_id} {v.status} ({v.conf:.0%})'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(vis, (x1, y1-th-8), (x1+tw+4, y1), color, -1)
        cv2.putText(vis, label, (x1+2, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1)
    
    # Draw pathfinding route
    if path and len(path) > 1:
        h, w = vis.shape[:2]
        cell_h, cell_w = h // grid_size, w // grid_size
        for i in range(len(path)-1):
            p1 = (path[i][1]*cell_w + cell_w//2, path[i][0]*cell_h + cell_h//2)
            p2 = (path[i+1][1]*cell_w + cell_w//2, path[i+1][0]*cell_h + cell_h//2)
            cv2.line(vis, p1, p2, (255, 255, 0), 2)
        # Start marker
        sp = (path[0][1]*cell_w + cell_w//2, path[0][0]*cell_h + cell_h//2)
        cv2.circle(vis, sp, 8, (0, 255, 0), -1)
        # End marker
        ep = (path[-1][1]*cell_w + cell_w//2, path[-1][0]*cell_h + cell_h//2)
        cv2.circle(vis, ep, 8, (0, 0, 255), -1)
    
    return vis

print('visualize_result() ready')

## 13. Demo — Jalankan Pipeline

In [ ]:
# === DEMO: Load sample image(s) ===
# Ganti path di bawah dengan gambar test dari dataset SARD
# Untuk demo optical flow, gunakan minimal 2 frame berurutan

import glob

# Cari test images dari SARD dataset
test_patterns = [
    '../data/kaggle/search-and-rescue/test/images/*.jpg',
    '../data/kaggle/search-and-rescue/test/images/*.png',
]
test_images = []
for pat in test_patterns:
    test_images.extend(sorted(glob.glob(pat)))

if not test_images:
    print('WARNING: No test images found. Using synthetic dummy frames for demo.')
    # Fallback: synthetic frames
    frames = [np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8) for _ in range(2)]
else:
    # Ambil 2 frame untuk demo optical flow
    sample_paths = test_images[:2]
    frames = [cv2.imread(p) for p in sample_paths]
    for p, f in zip(sample_paths, frames):
        print(f'Loaded: {Path(p).name} — {f.shape}')

print(f'\nTotal frames for pipeline: {len(frames)}')

In [ ]:
# === RUN FULL PIPELINE ===
result = run_pipeline(frames, sar_start=(0, 0), grid_size=50)

In [ ]:
# === TIMING SUMMARY ===
import statistics

print('\n=== LATENCY REPORT ===')
for stage, lats in result['timings'].items():
    if lats:
        avg = statistics.fmean(lats)
        print(f'  {stage:15s}: mean={avg:.4f}s  (n={len(lats)})')

print(f'\n=== TRIAGE SUMMARY ===')
print(json.dumps(result['world_map'], indent=2))

In [ ]:
# === VISUALISASI HASIL ===
# Tampilkan frame terakhir dengan overlay detection + terrain + path

last_frame = frames[-1]
class_map, _ = segment_terrain(last_frame)

# Collect victims from result
victims = []
for vd in result['victims']:
    victims.append(Victim(**{k: v for k, v in vd.items() if k in Victim.__dataclass_fields__}))

# Get first pathfinding result for visualization
path = None
if result['pathfinding']:
    # Reconstruct full path for first victim
    grid = build_traversability_grid(class_map, 50)
    first_pf = result['pathfinding'][0]
    h, w = last_frame.shape[:2]
    v = next(v for v in victims if v.victim_id == first_pf['victim_id'])
    cx = (v.bbox[0] + v.bbox[2]) / 2
    cy = (v.bbox[1] + v.bbox[3]) / 2
    goal = (min(int(cy/h*50), 49), min(int(cx/w*50), 49))
    path = astar(grid, (0, 0), goal)

vis = visualize_result(last_frame, victims, class_map, path)

# Save
out_path = '../results/pipeline_demo_output.jpg'
cv2.imwrite(out_path, vis)
print(f'Saved visualization to {out_path}')

# Display in notebook
from IPython.display import display, Image as IPImage
_, buf = cv2.imencode('.jpg', vis)
display(IPImage(data=buf.tobytes()))

In [ ]:
# === SAVE FULL REPORT ===
report_path = '../results/pipeline_inference_report.json'
with open(report_path, 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f'Full report saved to {report_path}')

## 14. Constraint Final Validation

Ringkasan kepatuhan constraint Track A:

In [ ]:
det_sz = os.path.getsize(DET_MODEL_PATH) / 1024**2
seg_sz = os.path.getsize(SEG_MODEL_PATH) / 1024**2
total_sz = det_sz + seg_sz

avg_det = statistics.fmean(result['timings']['detection']) if result['timings']['detection'] else 0
avg_seg = statistics.fmean(result['timings']['segmentation']) if result['timings']['segmentation'] else 0

print('=' * 60)
print('  CONSTRAINT VALIDATION — Track A: The Edge Vision')
print('=' * 60)
print(f'  C-A1  Model Size    : {total_sz:.3f} MB / 50.0 MB    {"PASS" if total_sz <= 50 else "FAIL"}')
print(f'        - Detection   : {det_sz:.3f} MB')
print(f'        - Segmentation: {seg_sz:.3f} MB')
print(f'  C-A2  CPU-only      : ONNX Runtime CPUExecutionProvider  PASS')
print(f'  C-A3  Latency/sample: Det {avg_det:.3f}s + Seg {avg_seg:.3f}s = {avg_det+avg_seg:.3f}s / 3.0s  {"PASS" if avg_det+avg_seg <= 3 else "FAIL"}')
print(f'  C-A4  Framework     : ONNX Runtime {ort.__version__}  PASS')
print(f'  C-A5  Offline       : No network calls  PASS')
print('=' * 60)